In [ ]:
import pylab as pl
import numpy as np
import scipy as sp
import scipy.optimize as spo
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import csv
import tensorflow as tf
from tensorflow import keras

In [ ]:
#train the LSTM
#begin by setting up two batches for training data
#half of the x training data is linear
#a 'zero' corresponds to linear

labels = np.arange(30)
batch_y_1 = np.full(250, 0)
batch_y_2 = np.full(250,1)

#generate dataframes for the x data sets
df_1 = pd.read_csv('np_lin_100.csv', names=labels)
df_2 = pd.read_csv('np_circ_100.csv', names=labels)

frames_x = [df_1, df_2]
batch_x = pd.concat(frames_x, ignore_index=True)
new = batch_x.iloc[:, :71]

#generate dataframes for the y data sets
df_3 = pd.DataFrame(data=batch_y_1)
df_4 = pd.DataFrame(data=batch_y_2)

frames_y = [df_3, df_4]
batch_y = pd.concat(frames_y, ignore_index=True)

#set up the training and testing split, 80-20 rule
X_train, X_test, Y_train, Y_test = train_test_split(batch_x, batch_y, test_size=0.2, random_state=21)

#scale the x data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#generate the tensors we need for pytorch!
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.float32)

#first, convert the x training data into a tensor
#the y training data becomes categorical
#batch_x_train = tf.convert_to_tensor(batch_x_train, dtype=tf.float32) 
#batch_y_train = tf.keras.utils.to_categorical(batch_y_train, num_classes=2)
#batch_y_val = batch_y_train

#convert it to a tensor
#batch_x_val = tf.convert_to_tensor(batch_x_val, dtype=tf.float32)

#then set the training parameters
lr = 1e-2
num_nodes = 6
num_classes = 2
num_epochs = 400
timesteps = 71
tf.keras.backend.clear_session()
X = tf.keras.Input(name='X', shape=[X_train_tensor.shape[1],1], dtype=tf.dtypes.float32)
lstm_output = tf.keras.layers.LSTM(num_nodes)(X)
prediction = tf.keras.layers.Dense(num_classes)(lstm_output)
opt = tf.keras.optimizers.Adam(learning_rate=lr)
model = keras.Model(inputs=X, outputs=prediction)
model.compile(optimizer=opt, loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),metrics=['accuracy'])
model.fit(x=X_train_tensor, y=Y_train_tensor, epochs=num_epochs, validation_data=(X_test_tensor,Y_test_tensor))

In [ ]:
#since we have training and validation sets,
#we can proceed with a testing set

batch_x_validate = []
df_3 = pd.read_csv('np_lin_300.csv')
for index, rows in df_3.iterrows():
    list_3 = rows
    batch_x_validate.append(list_3)
    
df_4 = pd.read_csv('np_circ_300.csv')
for index, rows in df_4.iterrows():
    list_4 = rows
    batch_x_validate.append(list_4)
    
batch_x_validate = tf.convert_to_tensor(batch_x_validate, dtype=tf.float32)

In [ ]:
#next, we generate a confusion matrix
y_prediction = model.predict(batch_x_test)
y_prediction = np.argmax(y_prediction, axis=1)
y_test = np.argmax(batch_y_train, axis=1)
result = tf.math.confusion_matrix(y_test, y_prediction, num_classes=2, weights=None)
labels = ["linear", "circular"]
cm = confusion_matrix(y_test, y_prediction)

In [ ]:
#create a display to evaluate the results
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues)
plt.show()